# Gold Layer

## Gold Publish

Purpose:
- Read the standardized Silver dataset
- Persist it as Delta in the Gold layer
- Validate the result

## Glue Catalog registration happens outside this notebook

This notebook only writes the Gold Delta table -- it does
**not** register it in the Glue Catalog.

Databricks Free Edition (the workspace this project targets)
only runs serverless compute hosted in Databricks' own AWS
account. There is no instance profile, no cross-account IAM
role trust, no secret scope configured -- `boto3`'s default
credential chain (env vars, `~/.aws/credentials`, instance
metadata) finds nothing from inside this notebook. The only
way to make `GlueCatalogProvider` work here would be a
long-lived AWS access key injected via a Databricks secret
scope -- the same kind of static, long-lived credential this
project has deliberately avoided everywhere else (no static
`DATABRICKS_TOKEN`, no hardcoded AWS keys).

Instead, the Glue registration is planned to run on the
Airflow side (where the real local AWS credential chain
already works today, the same one the `real_aws` integration
tests use): after `full_pipeline.yml`'s `gold` task finishes,
Airflow triggers a `GoldCatalogRegistrationStage` (same
pattern as the existing `CatalogPublishingStage` /
`GlueCatalogProvider`, already tested against real Glue) that
reads the Delta table's schema and registers it. See
`docs/architecture/roadmap-next-steps.md` -- not implemented
yet, out of scope for this notebook.

## Imports

In [ ]:
from data_platform.compute.delta_io import read_delta, write_delta
from data_platform.compute.spark import get_spark
from data_platform.storage.config import StorageConfig
from integrations.databricks.runtime.parameters import get_parameter

## Parameters

In [ ]:
entity = get_parameter("entity", default="customers")

## Spark Session

In [ ]:
spark = get_spark("Gold Publish")

## Read Silver

In [ ]:
df = read_delta(spark, StorageConfig.silver(entity))

## Write Gold

In [ ]:
write_delta(df, StorageConfig.gold(entity), mode="overwrite")

## Validation

In [ ]:
gold_df = read_delta(spark, StorageConfig.gold(entity))

print(f"Total records: {gold_df.count()}")
gold_df.printSchema()